# 14. Log-Mel CNN — Unseen-Generator Generalization

목표: **MusicGen**과 **Udio**를 각각 학습/검증에서 완전히 제외한 뒤, 해당 generator를 처음 보는 상황에서 Log-Mel CNN의 일반화 성능을 측정한다.

핵심 원칙:

- `FAKE = 1`, `REAL = 0`
- 기존 `original_audio` 기준 Train / Val / Test split은 그대로 유지
- Holdout generator의 FAKE는 **Train과 Val에서 0개**여야 함
- Unseen Test = Test REAL 전체 + Holdout generator FAKE만
- Validation EER threshold를 한 번 정하고 Test에 고정 적용
- Segment score와 Track 평균 score를 모두 평가
- **13번에서 만든 Log-Mel cache를 그대로 재사용**하므로 Log-Mel을 다시 생성하지 않음
- CNN 구조/학습 설정은 13번 in-domain CNN과 동일하게 유지

비교 대상:

1. CNN in-domain vs CNN unseen-generator
2. RBF-SVM unseen-generator vs Log-Mel CNN unseen-generator


## 1. 경로 / 라이브러리 / Device 설정

In [ ]:
from pathlib import Path
import gc
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)

PROJECT_ROOT = Path(
    "/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project"
)

SEGMENT_PATH = PROJECT_ROOT / "data/metadata/segment_manifest_10s.csv"
LOGMEL_DIR = PROJECT_ROOT / "data/processed/logmel"
LOGMEL_PATH = LOGMEL_DIR / "logmel_10s_float16.npy"
DONE_PATH = LOGMEL_DIR / "logmel_10s_done.npy"
INDEX_PATH = LOGMEL_DIR / "logmel_10s_index.csv"

RESULT_DIR = PROJECT_ROOT / "results/cnn_unseen_generator"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints/cnn_unseen_generator"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# 13번 CNN 결과: in-domain subgroup 비교용
CNN_TEST_TRACK_PATH = PROJECT_ROOT / "results/cnn/test_track_predictions.csv"
CNN_THRESHOLD_PATH = PROJECT_ROOT / "results/cnn/cnn_thresholds.json"

# 11번 Handcrafted unseen 결과: 모델 비교용
SVM_UNSEEN_PATH = PROJECT_ROOT / "results/unseen_generator/rbf_svm_track_unseen_summary.csv"

HOLDOUT_GENERATORS = ["musicgen", "udio"]
RANDOM_STATE = 42
BATCH_SIZE = 32
MAX_EPOCHS = 20
PATIENCE = 4

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Python/Torch device check")
print("PyTorch:", torch.__version__)
print("Device :", DEVICE)
print("Segment:", SEGMENT_PATH)
print("Log-Mel:", LOGMEL_PATH)
print("Holdout:", HOLDOUT_GENERATORS)


## 2. 재현성 함수

In [ ]:
def seed_everything(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
print("Random seed:", RANDOM_STATE)


## 3. Segment manifest + Log-Mel cache QC

In [ ]:
segments = pd.read_csv(SEGMENT_PATH).reset_index(drop=True)

if not LOGMEL_PATH.exists():
    raise FileNotFoundError(
        "13번에서 만든 Log-Mel cache가 없습니다: " + str(LOGMEL_PATH)
    )

logmel_cache = np.load(LOGMEL_PATH, mmap_mode="r")

if not DONE_PATH.exists():
    raise FileNotFoundError(DONE_PATH)

done_mask = np.load(DONE_PATH)

print("Segment rows :", len(segments))
print("Cache shape  :", logmel_cache.shape)
print("Cache dtype  :", logmel_cache.dtype)
print("Completed    :", int(done_mask.sum()), "/", len(done_mask))

assert len(segments) == 10077
assert logmel_cache.shape == (10077, 128, 1001)
assert bool(done_mask.all())

# 13번 cache index와 현재 manifest 순서가 같은지 확인
if INDEX_PATH.exists():
    cache_index = pd.read_csv(INDEX_PATH)
    same_order = (
        len(cache_index) == len(segments)
        and cache_index["segment_id"].astype(str).tolist()
            == segments["segment_id"].astype(str).tolist()
    )
    print("Cache index alignment:", same_order)
    assert same_order
else:
    print("INDEX_PATH가 없어 segment_id 순서 검증은 생략합니다.")

print("Log-Mel reuse QC PASS: True")


## 4. Holdout 데이터 구성 함수

In [ ]:
def build_holdout_data(df, holdout_generator):
    train = df[
        (df["split"] == "train")
        & (
            (df["label"] == "REAL")
            | (
                (df["label"] == "FAKE")
                & (df["generator"] != holdout_generator)
            )
        )
    ].copy()

    val = df[
        (df["split"] == "val")
        & (
            (df["label"] == "REAL")
            | (
                (df["label"] == "FAKE")
                & (df["generator"] != holdout_generator)
            )
        )
    ].copy()

    unseen_test = df[
        (df["split"] == "test")
        & (
            (df["label"] == "REAL")
            | (
                (df["label"] == "FAKE")
                & (df["generator"] == holdout_generator)
            )
        )
    ].copy()

    # 참고용: holdout을 제외한 나머지 seen generator test
    seen_test = df[
        (df["split"] == "test")
        & (
            (df["label"] == "REAL")
            | (
                (df["label"] == "FAKE")
                & (df["generator"] != holdout_generator)
            )
        )
    ].copy()

    return train, val, unseen_test, seen_test


## 5. Holdout 구조 QC — 반드시 먼저 확인

In [ ]:
EXPECTED_COUNTS = {
    "musicgen": {
        "train_rows": 6352,
        "val_rows": 1409,
        "unseen_test_rows": 270,
        "seen_test_rows": 1437,
    },
    "udio": {
        "train_rows": 6355,
        "val_rows": 1394,
        "unseen_test_rows": 279,
        "seen_test_rows": 1428,
    },
}

structure_rows = []

for holdout in HOLDOUT_GENERATORS:
    train_h, val_h, unseen_h, seen_h = build_holdout_data(
        segments, holdout
    )

    row = {
        "holdout_generator": holdout,
        "train_rows": len(train_h),
        "val_rows": len(val_h),
        "unseen_test_rows": len(unseen_h),
        "seen_test_rows": len(seen_h),
        "train_holdout_fake": int(
            ((train_h["label"] == "FAKE") & (train_h["generator"] == holdout)).sum()
        ),
        "val_holdout_fake": int(
            ((val_h["label"] == "FAKE") & (val_h["generator"] == holdout)).sum()
        ),
        "unseen_holdout_fake_segments": int(
            ((unseen_h["label"] == "FAKE") & (unseen_h["generator"] == holdout)).sum()
        ),
        "train_original_audio": train_h["original_audio"].nunique(),
        "val_original_audio": val_h["original_audio"].nunique(),
        "test_original_audio": unseen_h["original_audio"].nunique(),
        "unseen_real_segments": int((unseen_h["label"] == "REAL").sum()),
        "unseen_fake_segments": int((unseen_h["label"] == "FAKE").sum()),
        "unseen_real_tracks": unseen_h.loc[unseen_h["label"] == "REAL", "track_sample_id"].nunique(),
        "unseen_fake_tracks": unseen_h.loc[unseen_h["label"] == "FAKE", "track_sample_id"].nunique(),
    }
    structure_rows.append(row)

holdout_structure = pd.DataFrame(structure_rows)
display(holdout_structure)

for _, r in holdout_structure.iterrows():
    h = r["holdout_generator"]
    exp = EXPECTED_COUNTS[h]
    assert int(r["train_rows"]) == exp["train_rows"]
    assert int(r["val_rows"]) == exp["val_rows"]
    assert int(r["unseen_test_rows"]) == exp["unseen_test_rows"]
    assert int(r["seen_test_rows"]) == exp["seen_test_rows"]
    assert int(r["train_holdout_fake"]) == 0
    assert int(r["val_holdout_fake"]) == 0

print("Holdout Structure QC PASS: True")


## 6. Dataset / DataLoader

In [ ]:
class LogMelDataset(Dataset):
    def __init__(self, metadata, memmap_path):
        # reset_index(drop=False)의 index가 전체 segment_manifest row index
        self.metadata = metadata.reset_index(drop=False).copy()
        self.data = np.load(memmap_path, mmap_mode="r")

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        original_index = int(row["index"])

        x = np.asarray(
            self.data[original_index],
            dtype=np.float32,
        )
        x = torch.from_numpy(x.copy()).unsqueeze(0)

        y = torch.tensor(
            float(row["label_id"]),
            dtype=torch.float32,
        )

        return x, y, original_index


def make_loader(metadata, shuffle=False):
    ds = LogMelDataset(metadata, LOGMEL_PATH)
    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        drop_last=False,
    )
    return ds, loader


## 7. CNN 구조 — 13번과 동일

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class LogMelCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 16),
            ConvBlock(16, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x.squeeze(1)


tmp_model = LogMelCNN()
total_params = sum(p.numel() for p in tmp_model.parameters())
print("Total params:", total_params)
assert total_params == 294321
assert total_params < 2_000_000
del tmp_model


## 8. 평가 함수

In [ ]:
def find_eer_threshold(y_true, scores):
    fpr, tpr, thresholds = roc_curve(y_true, scores, pos_label=1)
    fnr = 1.0 - tpr
    valid = np.isfinite(thresholds)

    fpr = fpr[valid]
    fnr = fnr[valid]
    thresholds = thresholds[valid]

    idx = np.argmin(np.abs(fpr - fnr))

    return {
        "eer": float((fpr[idx] + fnr[idx]) / 2.0),
        "threshold": float(thresholds[idx]),
        "fpr_at_eer": float(fpr[idx]),
        "fnr_at_eer": float(fnr[idx]),
    }


def evaluate_scores(y_true, scores, threshold):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    y_pred = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    eer_info = find_eer_threshold(y_true, scores)

    return {
        "n_total": len(y_true),
        "n_real": int((y_true == 0).sum()),
        "n_fake": int((y_true == 1).sum()),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "pr_auc": float(average_precision_score(y_true, scores)),
        "eer": float(eer_info["eer"]),
        "threshold_used": float(threshold),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "real_fpr": float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan,
        "fake_miss_rate": float(fn / (fn + tp)) if (fn + tp) > 0 else np.nan,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def make_track_scores(metadata, scores):
    temp = metadata[
        [
            "track_sample_id",
            "original_audio",
            "label",
            "label_id",
            "genre",
            "generator",
            "split",
        ]
    ].copy()
    temp["score"] = scores

    return (
        temp.groupby("track_sample_id", as_index=False)
        .agg(
            original_audio=("original_audio", "first"),
            label=("label", "first"),
            label_id=("label_id", "first"),
            genre=("genre", "first"),
            generator=("generator", "first"),
            split=("split", "first"),
            segment_count=("score", "size"),
            score=("score", "mean"),
        )
    )


@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    all_scores, all_labels, all_indices = [], [], []

    for x, y, original_idx in loader:
        x = x.to(DEVICE)
        logits = model(x)
        scores = torch.sigmoid(logits).cpu().numpy()

        all_scores.append(scores)
        all_labels.append(y.numpy())
        all_indices.append(original_idx.numpy())

    return (
        np.concatenate(all_labels),
        np.concatenate(all_scores),
        np.concatenate(all_indices),
    )


## 9. 단일 Holdout CNN 학습 함수

In [ ]:
def run_cnn_holdout(holdout):
    print("\n" + "=" * 90)
    print("HOLDOUT GENERATOR:", holdout)
    print("=" * 90)

    seed_everything(RANDOM_STATE)

    train_meta, val_meta, unseen_meta, seen_meta = build_holdout_data(
        segments, holdout
    )

    train_ds, train_loader = make_loader(train_meta, shuffle=True)
    val_ds, val_loader = make_loader(val_meta, shuffle=False)
    unseen_ds, unseen_loader = make_loader(unseen_meta, shuffle=False)
    seen_ds, seen_loader = make_loader(seen_meta, shuffle=False)

    model = LogMelCNN().to(DEVICE)

    train_real = int((train_meta["label"] == "REAL").sum())
    train_fake = int((train_meta["label"] == "FAKE").sum())
    pos_weight = train_real / train_fake

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE)
    )
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-3, weight_decay=1e-4
    )

    checkpoint_path = CHECKPOINT_DIR / f"logmel_cnn_holdout_{holdout}_best.pt"
    history_rows = []
    best_val_track_eer = np.inf
    epochs_without_improvement = 0

    print("Train / Val / Unseen / Seen:", len(train_ds), len(val_ds), len(unseen_ds), len(seen_ds))
    print("Train REAL / FAKE:", train_real, train_fake)
    print("pos_weight:", pos_weight)

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        running_loss = 0.0
        seen_count = 0

        for x, y, _ in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            bs = x.size(0)
            running_loss += loss.item() * bs
            seen_count += bs

        train_loss = running_loss / seen_count

        val_y, val_scores, val_indices = predict_loader(model, val_loader)
        val_meta_ordered = segments.iloc[val_indices].reset_index(drop=True)

        val_seg_eer = find_eer_threshold(val_y, val_scores)
        val_track = make_track_scores(val_meta_ordered, val_scores)
        val_track_eer = find_eer_threshold(
            val_track["label_id"], val_track["score"]
        )
        val_auc = roc_auc_score(val_y, val_scores)
        elapsed = time.time() - epoch_start

        history_rows.append({
            "holdout_generator": holdout,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_segment_roc_auc": val_auc,
            "val_segment_eer": val_seg_eer["eer"],
            "val_track_eer": val_track_eer["eer"],
            "elapsed_sec": elapsed,
        })

        print(
            f"[{holdout}] Epoch {epoch:02d} | "
            f"loss={train_loss:.4f} | "
            f"val seg AUC={val_auc:.4f} | "
            f"val seg EER={val_seg_eer['eer']:.4f} | "
            f"val track EER={val_track_eer['eer']:.4f} | "
            f"{elapsed:.1f}s"
        )

        if val_track_eer["eer"] < best_val_track_eer - 1e-5:
            best_val_track_eer = val_track_eer["eer"]
            epochs_without_improvement = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": epoch,
                    "best_val_track_eer": best_val_track_eer,
                    "holdout_generator": holdout,
                    "pos_weight": pos_weight,
                },
                checkpoint_path,
            )
            print("  -> saved best model")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"  -> early stopping at epoch {epoch}")
                break

    history = pd.DataFrame(history_rows)
    history.to_csv(
        RESULT_DIR / f"{holdout}_training_history.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # Best checkpoint reload
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print(
        f"[{holdout}] BEST epoch={checkpoint['epoch']} | "
        f"Val Track EER={checkpoint['best_val_track_eer']:.4f}"
    )

    # Validation threshold 결정
    val_y, val_scores, val_indices = predict_loader(model, val_loader)
    val_meta_ordered = segments.iloc[val_indices].reset_index(drop=True)
    val_seg_eer = find_eer_threshold(val_y, val_scores)
    val_track = make_track_scores(val_meta_ordered, val_scores)
    val_track_eer = find_eer_threshold(val_track["label_id"], val_track["score"])

    segment_threshold = val_seg_eer["threshold"]
    track_threshold = val_track_eer["threshold"]

    result_rows = []
    prediction_outputs = {}

    for test_type, loader in [
        ("unseen_generator", unseen_loader),
        ("seen_generators", seen_loader),
    ]:
        y, scores, indices = predict_loader(model, loader)
        meta_ordered = segments.iloc[indices].reset_index(drop=True)

        seg_metrics = evaluate_scores(y, scores, segment_threshold)
        track_df = make_track_scores(meta_ordered, scores)
        track_metrics = evaluate_scores(
            track_df["label_id"], track_df["score"], track_threshold
        )

        result_rows.append({
            "holdout_generator": holdout,
            "model": "LogMelCNN",
            "level": "segment",
            "test_type": test_type,
            "val_eer": val_seg_eer["eer"],
            "best_epoch": int(checkpoint["epoch"]),
            **seg_metrics,
        })
        result_rows.append({
            "holdout_generator": holdout,
            "model": "LogMelCNN",
            "level": "track",
            "test_type": test_type,
            "val_eer": val_track_eer["eer"],
            "best_epoch": int(checkpoint["epoch"]),
            **track_metrics,
        })

        if test_type == "unseen_generator":
            seg_pred = meta_ordered[
                [
                    "segment_id",
                    "track_sample_id",
                    "original_audio",
                    "label",
                    "label_id",
                    "genre",
                    "generator",
                    "split",
                ]
            ].copy()
            seg_pred["cnn_score"] = scores
            seg_pred["threshold"] = segment_threshold
            seg_pred["cnn_pred"] = (scores >= segment_threshold).astype(int)

            track_pred = track_df.copy()
            track_pred["threshold"] = track_threshold
            track_pred["cnn_pred"] = (
                track_pred["score"] >= track_threshold
            ).astype(int)
            track_pred = track_pred.rename(columns={"score": "cnn_score"})

            prediction_outputs["segment"] = seg_pred
            prediction_outputs["track"] = track_pred

    threshold_info = {
        "holdout_generator": holdout,
        "segment_eer_threshold": float(segment_threshold),
        "track_eer_threshold": float(track_threshold),
        "segment_val_eer": float(val_seg_eer["eer"]),
        "track_val_eer": float(val_track_eer["eer"]),
        "best_epoch": int(checkpoint["epoch"]),
    }

    with open(
        RESULT_DIR / f"{holdout}_thresholds.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(threshold_info, f, indent=2, ensure_ascii=False)

    # 메모리 정리
    del model, optimizer, criterion
    gc.collect()
    if DEVICE.type == "mps" and hasattr(torch, "mps"):
        torch.mps.empty_cache()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return pd.DataFrame(result_rows), history, prediction_outputs


## 10. MusicGen / Udio Holdout 학습 실행

이 셀이 가장 오래 걸린다. 13번 실험에서 epoch당 약 78~82초였다면, 두 holdout을 합쳐 대략 **30~55분 정도**를 예상하면 된다. Early stopping 시 더 짧아질 수 있다.


In [ ]:
all_results = []
all_histories = []
all_predictions = {}

for holdout in HOLDOUT_GENERATORS:
    result_df, history_df, pred_dict = run_cnn_holdout(holdout)
    all_results.append(result_df)
    all_histories.append(history_df)
    all_predictions[holdout] = pred_dict

cnn_unseen_results = pd.concat(all_results, ignore_index=True)
training_history_all = pd.concat(all_histories, ignore_index=True)

print("\nTraining complete.")
print("Result rows:", len(cnn_unseen_results))


## 11. 학습 곡선 확인

In [ ]:
display(training_history_all)

for holdout in HOLDOUT_GENERATORS:
    h = training_history_all[
        training_history_all["holdout_generator"] == holdout
    ]
    plt.figure(figsize=(8, 5))
    plt.plot(h["epoch"], h["val_segment_eer"], marker="o", label="segment EER")
    plt.plot(h["epoch"], h["val_track_eer"], marker="o", label="track EER")
    plt.xlabel("epoch")
    plt.ylabel("EER")
    plt.title(f"CNN Validation EER — holdout {holdout}")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 12. 핵심 결과 — CNN Track-level Unseen Generator

In [ ]:
unseen_only = cnn_unseen_results[
    cnn_unseen_results["test_type"] == "unseen_generator"
].copy()

cnn_track_unseen = unseen_only[
    unseen_only["level"] == "track"
].copy()

cols = [
    "holdout_generator",
    "best_epoch",
    "n_real",
    "n_fake",
    "roc_auc",
    "eer",
    "balanced_accuracy",
    "macro_f1",
    "real_fpr",
    "fake_miss_rate",
    "threshold_used",
]

display(cnn_track_unseen[cols].round(4))


## 13. Segment vs Track — Unseen 환경

In [ ]:
display(
    unseen_only[
        [
            "holdout_generator",
            "level",
            "n_real",
            "n_fake",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ]
    .sort_values(["holdout_generator", "level"])
    .round(4)
)


## 14. RBF-SVM vs Log-Mel CNN — Unseen Generator

In [ ]:
if SVM_UNSEEN_PATH.exists():
    svm_unseen = pd.read_csv(SVM_UNSEEN_PATH).copy()

    svm_unseen["model"] = "RBF-SVM"
    svm_compare = svm_unseen[
        [
            "holdout_generator",
            "model",
            "n_real",
            "n_fake",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ].copy()

    cnn_compare = cnn_track_unseen[
        [
            "holdout_generator",
            "model",
            "n_real",
            "n_fake",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ].copy()

    model_comparison = pd.concat(
        [svm_compare, cnn_compare],
        ignore_index=True,
    ).sort_values(["holdout_generator", "model"])

    display(model_comparison.round(4))

    pivot_auc = model_comparison.pivot(
        index="holdout_generator",
        columns="model",
        values="roc_auc",
    ).reset_index()
    if {"RBF-SVM", "LogMelCNN"}.issubset(pivot_auc.columns):
        pivot_auc["cnn_minus_svm_auc"] = (
            pivot_auc["LogMelCNN"] - pivot_auc["RBF-SVM"]
        )
        print("\nROC-AUC difference (CNN - SVM)")
        display(pivot_auc.round(4))
else:
    print("SVM unseen summary not found:", SVM_UNSEEN_PATH)


## 15. CNN In-domain vs Unseen — MusicGen / Udio

In [ ]:
cnn_in_domain_compare = None

if CNN_TEST_TRACK_PATH.exists() and CNN_THRESHOLD_PATH.exists():
    in_domain_pred = pd.read_csv(CNN_TEST_TRACK_PATH)

    with open(CNN_THRESHOLD_PATH, "r", encoding="utf-8") as f:
        base_threshold_info = json.load(f)

    base_track_threshold = float(base_threshold_info["track_eer_threshold"])

    in_domain_rows = []
    for holdout in HOLDOUT_GENERATORS:
        subset = in_domain_pred[
            (in_domain_pred["label"] == "REAL")
            | (
                (in_domain_pred["label"] == "FAKE")
                & (in_domain_pred["generator"] == holdout)
            )
        ].copy()

        metrics = evaluate_scores(
            subset["label_id"],
            subset["cnn_score"],
            base_track_threshold,
        )

        in_domain_rows.append({
            "holdout_generator": holdout,
            "in_domain_roc_auc": metrics["roc_auc"],
            "in_domain_eer": metrics["eer"],
            "in_domain_balanced_accuracy": metrics["balanced_accuracy"],
            "in_domain_macro_f1": metrics["macro_f1"],
            "in_domain_real_fpr": metrics["real_fpr"],
            "in_domain_fake_miss_rate": metrics["fake_miss_rate"],
        })

    cnn_in_domain = pd.DataFrame(in_domain_rows)

    cnn_in_domain_compare = cnn_track_unseen.merge(
        cnn_in_domain,
        on="holdout_generator",
        how="left",
        validate="one_to_one",
    )

    cnn_in_domain_compare["roc_auc_drop"] = (
        cnn_in_domain_compare["in_domain_roc_auc"]
        - cnn_in_domain_compare["roc_auc"]
    )
    cnn_in_domain_compare["eer_increase"] = (
        cnn_in_domain_compare["eer"]
        - cnn_in_domain_compare["in_domain_eer"]
    )
    cnn_in_domain_compare["fake_miss_increase"] = (
        cnn_in_domain_compare["fake_miss_rate"]
        - cnn_in_domain_compare["in_domain_fake_miss_rate"]
    )

    display(
        cnn_in_domain_compare[
            [
                "holdout_generator",
                "in_domain_roc_auc",
                "roc_auc",
                "roc_auc_drop",
                "in_domain_eer",
                "eer",
                "eer_increase",
                "in_domain_fake_miss_rate",
                "fake_miss_rate",
                "fake_miss_increase",
            ]
        ].round(4)
    )
else:
    print("13번 CNN prediction/threshold 파일을 찾지 못해 in-domain 비교를 생략합니다.")


## 16. Seen-generator Mixed Test 참고

In [ ]:
seen_only = cnn_unseen_results[
    cnn_unseen_results["test_type"] == "seen_generators"
].copy()

display(
    seen_only[
        [
            "holdout_generator",
            "level",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ]
    .sort_values(["holdout_generator", "level"])
    .round(4)
)


## 17. Prediction / Metrics 저장

In [ ]:
for holdout, pred_dict in all_predictions.items():
    holdout_dir = RESULT_DIR / holdout
    holdout_dir.mkdir(parents=True, exist_ok=True)

    pred_dict["segment"].to_csv(
        holdout_dir / "logmelcnn_segment_unseen_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )
    pred_dict["track"].to_csv(
        holdout_dir / "logmelcnn_track_unseen_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )

cnn_unseen_results.to_csv(
    RESULT_DIR / "cnn_unseen_generator_metrics_all.csv",
    index=False,
    encoding="utf-8-sig",
)

cnn_track_unseen.to_csv(
    RESULT_DIR / "cnn_track_unseen_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

training_history_all.to_csv(
    RESULT_DIR / "training_history_all.csv",
    index=False,
    encoding="utf-8-sig",
)

if "model_comparison" in globals():
    model_comparison.to_csv(
        RESULT_DIR / "rbf_svm_vs_cnn_unseen_track.csv",
        index=False,
        encoding="utf-8-sig",
    )

if cnn_in_domain_compare is not None:
    cnn_in_domain_compare.to_csv(
        RESULT_DIR / "cnn_in_domain_vs_unseen.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("Saved results to:", RESULT_DIR)


## 18. 최종 QC

In [ ]:
qc_rows = []

for holdout in HOLDOUT_GENERATORS:
    s = holdout_structure[
        holdout_structure["holdout_generator"] == holdout
    ].iloc[0]

    subset = unseen_only[
        unseen_only["holdout_generator"] == holdout
    ]

    ckpt = CHECKPOINT_DIR / f"logmel_cnn_holdout_{holdout}_best.pt"

    qc_rows.append({
        "holdout_generator": holdout,
        "train_holdout_fake": int(s["train_holdout_fake"]),
        "val_holdout_fake": int(s["val_holdout_fake"]),
        "unseen_test_fake_segments": int(s["unseen_holdout_fake_segments"]),
        "unseen_real_tracks": int(s["unseen_real_tracks"]),
        "unseen_fake_tracks": int(s["unseen_fake_tracks"]),
        "result_rows": len(subset),
        "nan_roc_auc": int(subset["roc_auc"].isna().sum()),
        "nonfinite_threshold": int((~np.isfinite(subset["threshold_used"])).sum()),
        "checkpoint_exists": ckpt.exists(),
    })

qc_summary = pd.DataFrame(qc_rows)
display(qc_summary)

expected_fake_tracks = {"musicgen": 45, "udio": 48}

cnn_unseen_qc_pass = (
    bool(done_mask.all())
    and (qc_summary["train_holdout_fake"] == 0).all()
    and (qc_summary["val_holdout_fake"] == 0).all()
    and (qc_summary["unseen_test_fake_segments"] > 0).all()
    and (qc_summary["unseen_real_tracks"] == 45).all()
    and all(
        int(
            qc_summary.loc[
                qc_summary["holdout_generator"] == h,
                "unseen_fake_tracks",
            ].iloc[0]
        ) == n
        for h, n in expected_fake_tracks.items()
    )
    and (qc_summary["result_rows"] == 2).all()
    and (qc_summary["nan_roc_auc"] == 0).all()
    and (qc_summary["nonfinite_threshold"] == 0).all()
    and qc_summary["checkpoint_exists"].all()
    and total_params == 294321
)

print("===== FINAL RESULT =====")
print("CNN Unseen Generator Core QC PASS:", cnn_unseen_qc_pass)


## 다음 단계

이 실험이 끝나면 **15. CNN MP3 Robustness**로 넘어간다.

최종 핵심 비교는 다음과 같다.

- In-domain: RBF-SVM vs Log-Mel CNN
- Unseen generator: RBF-SVM vs Log-Mel CNN
- Compression: Original / MP3 128k / MP3 64k

특히 이번 14번에서는 다음 두 표가 가장 중요하다.

1. `RBF-SVM vs Log-Mel CNN — Unseen Generator`
2. `CNN In-domain vs Unseen — MusicGen / Udio`


## 최신 실행 결과 요약 (2026-09-13)

Log-Mel CNN Track-level unseen-generator 결과다.

| Holdout generator | ROC-AUC | EER | Balanced Accuracy | FAKE Miss Rate |
|---|---:|---:|---:|---:|
| MusicGen | 0.8681 | 0.2667 | 0.7333 | 0.4000 |
| Udio | 0.8227 | 0.2146 | 0.7653 | 0.2917 |

- MusicGen에서는 CNN이 RBF-SVM(0.6123)보다 크게 우수했다.
- Udio에서는 RBF-SVM(0.8542)이 CNN보다 소폭 높은 ROC-AUC를 보였다.
- 모델 우위가 generator마다 달라 단일 unseen generator만으로 일반화 성능을 단정할 수 없다.
- 결과를 `results/cnn_unseen_generator/`에 저장했다.

**최종 상태: CNN unseen-generator 실험 완료.**
